# Template selection

Picks the per-rib registration template (the patient whose 24 ribs are
*most central* across a random subsample) using Scalismo's
`TemplateBuilder`.  The chosen patient ID is written to
`<STL_PER_RIB>/template_id.txt` as a suggestion.  To put it to work, set
it as `template_id` in the preset: `RibRegistration` receives the PID via
`--template-pid`, and the pipeline driver writes the load-bearing
`template_id.txt` into the *registered* STL dir for the PCA stage.

**Iterating on the seed.**  If the suggested template looks wrong
(missing rib, severe asymmetry, segmentation artefact, …), bump
`SEED` in the next cell and re-run the notebook (`Run All`).  Each
run overwrites `template_id.txt` and re-renders the 3D viewer.

**Caveat.**  `RibRegistration` skips any rib whose output STL already
exists (`skipExisting`, on unless `--no-skip-existing` is passed), and it
does not check which template produced them.  Re-running against a new
template into a populated output directory therefore keeps the old
registrations silently.  Purge the registration output dir, or point at a
different one.


In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

NOTEBOOK_DIR = Path().resolve()
ROOT         = NOTEBOOK_DIR.parent
sys.path.insert(0, str(ROOT / 'src'))

import numpy as np
import plotly.graph_objects as go
import pyvista as pv
import yaml

from settings import apply_publication_style
from utils.colors import cmap as _cmap
from utils.paths import extracted_stl_dir
from utils.run_dir import patient_stl_dir
from ssm.scalismo import scalismo_run

apply_publication_style()

# ── Per-side colour for the 3D viewer ────────────────────────────────────
# Sample a mid-tone from each per-side ramp (rib_left = teal, rib_right =
# reds).
def _mid_hex(cm, t: float = 0.65) -> str:
    r, g, b, _ = cm(t)
    return "#{:02X}{:02X}{:02X}".format(int(r * 255), int(g * 255), int(b * 255))

SIDE_COLOR = {
    'Left':  _mid_hex(_cmap('rib_left')),
    'Right': _mid_hex(_cmap('rib_right')),
}

# ── Iterate by changing this seed and re-running the notebook ────────────
SEED        = 7
SAMPLE_SIZE = 500

# ── STL input dir — derived from the preset's `name` + `paths.root`, the
#    same way run_pipeline.py derives it (`paths.root` is the only path key
#    a preset carries).
PRESET_PATH = ROOT / 'presets' / 'example.yaml'
_preset = yaml.safe_load(PRESET_PATH.read_text()) or {}
_root   = (_preset.get('paths') or {}).get('root')
_name   = _preset.get('name')
if not _root or _root == 'REPLACE_ME_WITH_AN_ABSOLUTE_PATH':
    raise ValueError(
        f"{PRESET_PATH}: paths.root is missing or still the placeholder.  "
        f"Set it to an absolute path before running this notebook."
    )
if not _name:
    raise ValueError(f"{PRESET_PATH}: `name` is missing.")

STL_PER_RIB          = extracted_stl_dir(Path(_root).expanduser(), _name)
EXTRACT_TEMPLATE_TXT = STL_PER_RIB / 'template_id.txt'

RIB_LABELS = list(range(40, 52))
RIB_SIDES  = ['L', 'R']

print('Preset            :', PRESET_PATH)
print('Per-rib STL dir   :', STL_PER_RIB)
print('Template file     :', EXTRACT_TEMPLATE_TXT)
if EXTRACT_TEMPLATE_TXT.exists():
    print('Existing template :', EXTRACT_TEMPLATE_TXT.read_text().strip(),
          '(will be overwritten)')
else:
    print('Existing template : (none)')
print(f'SEED              : {SEED}   SAMPLE_SIZE: {SAMPLE_SIZE}')


## Run Scalismo TemplateBuilder

Pairwise mean-surface distance on a random subsample of `SAMPLE_SIZE`
patients.  The patient with the lowest aggregate centrality score
across all 24 rib identities wins.


In [ ]:
if not STL_PER_RIB.exists() or not any(STL_PER_RIB.glob('*/*/*.stl')):
    raise FileNotFoundError(
        f'No per-rib STLs found under {STL_PER_RIB}/<block>/<pid>/ — '
        f'run the mesh_extraction pipeline stage first '
        f'(or fix data_config.yaml).'
    )

sbt_cmd = (
    f'"runMain nako.ribs.TemplateBuilder'
    f' --input \\"{STL_PER_RIB}\\"'
    f' --outTxt \\"{EXTRACT_TEMPLATE_TXT}\\"'
    f' --sample {SAMPLE_SIZE}'
    f' --seed {SEED}"'
)
rc = scalismo_run(sbt_cmd)
if rc != 0:
    raise RuntimeError(f'TemplateBuilder exited with code {rc}')

if not EXTRACT_TEMPLATE_TXT.exists():
    raise RuntimeError(
        f'TemplateBuilder finished but {EXTRACT_TEMPLATE_TXT} was not '
        f'written — check sbt output above.'
    )

TEMPLATE_PID = EXTRACT_TEMPLATE_TXT.read_text().strip()
print(f'\nChosen template patient: {TEMPLATE_PID}')


## 3D viewer — chosen template's 24 ribs

Left ribs in teal, right ribs in red — mid-tones sampled from
`utils.colors.cmap('rib_left')` / `cmap('rib_right')`.  Drag to rotate,
scroll to zoom.


In [ ]:
from utils.rib_labels import display_from_seg

missing = []
traces  = []
all_pts = []

template_pdir = patient_stl_dir(STL_PER_RIB, TEMPLATE_PID)
for label in RIB_LABELS:
    for side in RIB_SIDES:
        stl_path = template_pdir / f'{TEMPLATE_PID}_rib{label}_{side}.stl'
        if not stl_path.exists():
            missing.append(stl_path.name)
            continue
        mesh = pv.read(str(stl_path))
        verts = np.asarray(mesh.points, dtype=np.float64)
        # PyVista stores faces as [3, i, j, k, 3, i, j, k, ...] for triangles.
        faces = np.asarray(mesh.faces, dtype=np.int64).reshape(-1, 4)[:, 1:]
        side_long  = 'Left' if side == 'L' else 'Right'
        display_id = display_from_seg(label, side)
        traces.append(go.Mesh3d(
            x=verts[:, 0], y=verts[:, 1], z=verts[:, 2],
            i=faces[:, 0], j=faces[:, 1], k=faces[:, 2],
            color=SIDE_COLOR[side_long],
            opacity=1.0,
            flatshading=False,
            lighting=dict(ambient=0.6, diffuse=0.7,
                          specular=0.3, roughness=0.5),
            lightposition=dict(x=1000, y=1000, z=1000),
            name=display_id,
            showlegend=False,
            hovertemplate=f'{display_id}<extra></extra>',
        ))
        all_pts.append(verts)

if missing:
    print(f'WARNING: {len(missing)} rib STL(s) missing for template patient '
          f'{TEMPLATE_PID}: {missing[:6]}{" ..." if len(missing) > 6 else ""}')
if not traces:
    raise RuntimeError(f'No rib STLs loaded for template {TEMPLATE_PID}')

# Anterior-right oblique camera, matching ssm.viewer's HTML viewer.
fig = go.Figure(data=traces)
fig.update_layout(
    scene=dict(
        xaxis=dict(visible=False),
        yaxis=dict(visible=False),
        zaxis=dict(visible=False),
        bgcolor='#fafafa',
        camera=dict(eye=dict(x=1.5, y=-1.8, z=0.5),
                    up=dict(x=0, y=0, z=-1)),
        aspectmode='data',
    ),
    margin=dict(l=0, r=0, t=40, b=0),
    paper_bgcolor='#fafafa',
    title=dict(text=f'Template patient {TEMPLATE_PID} — 24 per-rib STLs '
                    f'(SEED={SEED})',
               font=dict(size=14)),
    height=700,
)
fig.show()